In [1]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input

In [2]:
data = pd.read_csv("cleaned_heart.csv")

print("Shape:", data.shape)
data.head()

Shape: (920, 20)


,id,age,trestbps,chol,fbs,thalch,exang,oldpeak,num,sex_Male,dataset_Hungary,dataset_Switzerland,dataset_VA Long Beach,cp_atypical angina,cp_non-anginal,cp_typical angina,restecg_normal,restecg_st-t abnormality,slope_flat,slope_upsloping
0,1,63,145.0,233.0,True,150.0,False,2.3,0,True,False,False,False,False,False,True,False,False,False,False
1,2,67,160.0,286.0,False,108.0,True,1.5,2,True,False,False,False,False,False,False,False,False,True,False
2,3,67,120.0,229.0,False,129.0,True,2.6,1,True,False,False,False,False,False,False,False,False,True,False
3,4,37,130.0,250.0,False,187.0,False,3.5,0,True,False,False,False,False,True,False,True,False,False,False
4,5,41,130.0,204.0,False,172.0,False,1.4,0,False,False,False,False,True,False,False,False,False,False,True


In [3]:
data['risk'] = data['num'].apply(lambda x: 1 if x > 0 else 0)

data['risk'].value_counts()

risk
1    509
0    411
Name: count, dtype: int64

In [4]:
drop_cols = ['num', 'risk']

if 'id' in data.columns:
    drop_cols.append('id')

X = data.drop(columns=drop_cols)
y = data['risk']

print("Features:", X.columns.tolist())

Features: ['age', 'trestbps', 'chol', 'fbs', 'thalch', 'exang', 'oldpeak', 'sex_Male', 'dataset_Hungary', 'dataset_Switzerland', 'dataset_VA Long Beach', 'cp_atypical angina', 'cp_non-anginal', 'cp_typical angina', 'restecg_normal', 'restecg_st-t abnormality', 'slope_flat', 'slope_upsloping']


In [5]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Save scaler 🔥
joblib.dump(scaler, "scaler.pkl")

print("✅ Scaler saved")

✅ Scaler saved


In [6]:
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

In [7]:
def generate_sequences(data, labels, timesteps=15):
    X_seq = []
    y_seq = []

    for i in range(len(data)):
        row = data.iloc[i].values
        sequence = []

        for t in range(timesteps):
            step = row.copy()

            # progression
            step = step * (1 + t * 0.03)

            # noise
            step += np.random.normal(0, 0.01, size=row.shape)

            sequence.append(step)

        X_seq.append(sequence)
        y_seq.append(labels.iloc[i])

    return np.array(X_seq), np.array(y_seq)

X_lstm, y_lstm = generate_sequences(X_scaled, y)

print("Shape:", X_lstm.shape)

Shape: (920, 15, 18)


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X_lstm, y_lstm,
    test_size=0.2,
    random_state=42
)

In [9]:
model = Sequential()

model.add(Input(shape=(X_train.shape[1], X_train.shape[2])))
model.add(LSTM(64))
model.add(Dropout(0.3))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 64)                  │          21,248 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 21,313 (83.25 KB)

 Trainable params: 21,313 (83.25 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/15
19/19 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - accuracy: 0.6565 - loss: 0.6236 - val_accuracy: 0.8311 - val_loss: 0.5004
Epoch 2/15
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7908 - loss: 0.4708 - val_accuracy: 0.8311 - val_loss: 0.4121
Epoch 3/15
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.7976 - loss: 0.4317 - val_accuracy: 0.8378 - val_loss: 0.4092
Epoch 4/15
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8027 - loss: 0.4299 - val_accuracy: 0.8041 - val_loss: 0.4082
Epoch 5/15
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8027 - loss: 0.4187 - val_accuracy: 0.8311 - val_loss: 0.3975
Epoch 6/15
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8027 - loss: 0.4086 - val_accuracy: 0.8378 - val_loss: 0.4134
Epoch 7/15
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8095 - loss: 0.4089 - val_accuracy: 0.8311 - val_loss: 0.3918
Epoch 8/15
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.8214 - loss: 0.4005 - val_accuracy: 0.8446 - v

In [11]:
loss, acc = model.evaluate(X_test, y_test)

print("Test Accuracy:", acc)

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8370 - loss: 0.3762 
Test Accuracy: 0.8369565010070801


In [12]:
model.save("lstm_model_15.h5")

print("✅ Model saved")

✅ Model saved


In [13]:
print(os.listdir())

['.ipynb_checkpoints', 'cleaned_heart.csv', 'heart.csv', 'lstm_data.csv', 'lstm_model_15.h5', 'lstm_train.py', 'model.pkl', 'preprocess.py', 'scaler.pkl', 'train.py', 'Untitled.ipynb']


In [14]:
model = load_model("lstm_model_15.h5")
scaler = joblib.load("scaler.pkl")

print("✅ Loaded successfully")

✅ Loaded successfully


In [15]:
feature_names = X.columns.tolist()

In [16]:
user_df = pd.DataFrame([[
    45, 1, 2, 130, 250,
    0, 1, 150, 0,
    1.2, 2
]], columns=feature_names)

user_df

ValueError: 18 columns passed, passed data had 11 columns

In [17]:
feature_names = X.columns.tolist()
len(feature_names)

18

In [18]:
user_df = X.iloc[[0]]   # takes one full correct row
user_df

,age,trestbps,chol,fbs,thalch,exang,oldpeak,sex_Male,dataset_Hungary,dataset_Switzerland,dataset_VA Long Beach,cp_atypical angina,cp_non-anginal,cp_typical angina,restecg_normal,restecg_st-t abnormality,slope_flat,slope_upsloping
0,63,145.0,233.0,True,150.0,False,2.3,True,False,False,False,False,False,True,False,False,False,False


In [19]:
user_df['age'] = 50
user_df['chol'] = 200

C:\Users\R SAHANA\AppData\Local\Temp\ipykernel_25056\2685329364.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  user_df['age'] = 50
C:\Users\R SAHANA\AppData\Local\Temp\ipykernel_25056\2685329364.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  user_df['chol'] = 200


In [20]:
user_scaled = scaler.transform(user_df)

In [21]:
user_df = pd.DataFrame([[
    45, 1, 2, 130, 250,
    0, 1, 150, 0,
    1.2, 2
]], columns=feature_names)

user_df

ValueError: 18 columns passed, passed data had 11 columns